# Milestone 4: Final Results Gallery

Builds `results/milestone4/gallery/` from the held-out **test** split, with the models and settings from
`Integrated_Pipeline.ipynb` (run it first):

* **Part A:** 24 correctly classified single faces (6 per identity) and 8 misclassified ones (2 per identity),
  each error with a short analysis.
* **Part B:** 12 multi-face images with the pipeline's boxes and names.
* **Part C:** the gallery `README.md`.

Examples are picked by fixed rules, not by hand.

#### 0. Environment setup (local Jupyter or Google Colab)
Same cell as the other notebooks: on Colab it mounts Drive and installs the requirements, locally it finds the repo root.

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/rasmussenj03/IE7615-Group3-Project1.git"
COLAB_USE_DRIVE = True  # False -> clone into /content (outputs are lost when the runtime ends)
COLAB_REPO_DIR = ("/content/drive/MyDrive/IE7615-Group3-Project1" if COLAB_USE_DRIVE
                  else "/content/IE7615-Group3-Project1")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if COLAB_USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    if not Path(COLAB_REPO_DIR, "src").is_dir():
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        clone = subprocess.run(["git", "clone", "-q", REPO_URL.replace("https://", f"https://{token}@"),
                                COLAB_REPO_DIR], capture_output=True)
        if clone.returncode != 0:
            raise RuntimeError("git clone failed - check the GITHUB_TOKEN secret and repo access")
        subprocess.run(["git", "-C", COLAB_REPO_DIR, "remote", "set-url", "origin", REPO_URL])
    os.chdir(COLAB_REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path.cwd()
    while not (root / "src" / "pipeline.py").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)

PROJECT_ROOT = Path.cwd()

In [2]:
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

from src import detection_eval as de
from src.adapt_classifier import accuracy, collect_crops
from src.pipeline import CelebrityPipeline, predict_split, use_identity

GALLERY_DIR = PROJECT_ROOT / "results" / "milestone4" / "gallery"
for folder in ("correct", "misclassified", "multi_face"):
    (GALLERY_DIR / folder).mkdir(parents=True, exist_ok=True)
pipe = CelebrityPipeline()
print("Settings:", pipe.settings)

Settings: {'det_conf': 0.7, 'nms_iou': 0.7, 'imgsz': 640, 'identity': 'detector'}


---
## Part A: Single faces

In [3]:
# Classify every labelled test face
crops, faces = collect_crops("test")
acc, probs = accuracy(pipe.classifier, crops, faces, pipe.device)
faces["pred"], faces["conf"] = probs.argmax(1), probs.max(1)
faces["correct"] = faces["pred"] == faces["label"]
print(f"Test face accuracy: {acc:.3f} ({len(faces)} faces)")

Test face accuracy: 0.864 (543 faces)


In [4]:
# Correct: 6 per identity (3 original, 3 augmented), spread over the confidence range.
# Misclassified: the 2 most confident errors per true identity.
def spread(group, n=3):
    group = group.sort_values("conf")
    return group.iloc[np.linspace(0, len(group) - 1, n).round().astype(int)]


right = faces[faces["correct"]]
correct = pd.concat([spread(g) for _, by_id in right.groupby("label") for _, g in by_id.groupby("augmented")])
wrong = pd.concat([g.nlargest(2, "conf") for _, g in faces[~faces["correct"]].groupby("label")])
len(correct), len(wrong)

(24, 8)

In [5]:
# Short analysis of an error, from what can be measured on the crop
LOOKALIKES = [{"id_2970", "id_4422"}, {"id_7007", "id_4422"}]


def analyse(i):
    row, pixels = faces.loc[i], np.asarray(crops[i], dtype=float)
    true, pred = de.CLASS_NAMES[row["label"]], de.CLASS_NAMES[row["pred"]]
    w, h = Image.open(PROJECT_ROOT / "data" / "detection" / "images" / "test" / row["image"]).size
    causes = []
    if row["x1"] <= 1 or row["y1"] <= 1 or row["x2"] >= w - 1 or row["y2"] >= h - 1:
        causes.append("cut by the image border")
    if (pixels.max(axis=2) < 10).mean() > 0.05:
        causes.append("partly covered by a dropout hole")
    if np.abs(pixels[..., 0] - pixels[..., 2]).mean() < 2:
        causes.append("grey-scale copy")
    if row["y2"] - row["y1"] < 110:
        causes.append(f"small face ({row['y2'] - row['y1']:.0f} px)")
    text = ("Likely cause: " + ", ".join(causes) + ". ") if causes else "Clear, unaugmented face. "
    if {true, pred} in LOOKALIKES:
        text += f"{true} / {pred} is the look-alike pair also confused in Milestones 1 and 3. "
    text += "Confident error." if row["conf"] >= 0.8 else f"Low confidence ({row['conf']:.2f})."
    return text

In [6]:
# Save each face as a captioned tile (green = correct, red = wrong)
def save_tile(i, path):
    row, crop = faces.loc[i], crops[i]
    crop = crop.resize((round(crop.width * 160 / crop.height), 160))
    tile = Image.new("RGB", (max(crop.width, 190), 206), "white")
    tile.paste(crop, ((tile.width - crop.width) // 2, 0))
    draw = ImageDraw.Draw(tile)
    draw.rectangle([0, 160, tile.width, 206], fill="#1B7F3B" if row["correct"] else "#C62828")
    draw.text((4, 163), f"pred {de.CLASS_NAMES[row['pred']]} {row['conf']:.2f}", fill="white", font=de._font(13))
    draw.text((4, 183), f"true {de.CLASS_NAMES[row['label']]}", fill="white", font=de._font(13))
    tile.save(path)
    return tile


rows = []
for folder, picks in (("correct", correct), ("misclassified", wrong)):
    tiles = []
    for k, i in enumerate(picks.index, 1):
        r = faces.loc[i]
        file = f"{k:02d}_{Path(r['image']).stem}.jpg"
        tiles.append(save_tile(i, GALLERY_DIR / folder / file))
        rows.append(dict(folder=folder, file=file, true=de.CLASS_NAMES[r["label"]],
                         predicted=de.CLASS_NAMES[r["pred"]], confidence=round(float(r["conf"]), 2),
                         analysis=analyse(i) if folder == "misclassified" else ""))
    sheet = Image.new("RGB", (6 * 196, -(-len(tiles) // 6) * 212), "white")  # contact sheet, 6 per row
    for k, t in enumerate(tiles):
        sheet.paste(t, ((k % 6) * 196, (k // 6) * 212))
    sheet.save(GALLERY_DIR / f"{folder}.png")
single = pd.DataFrame(rows)
single[single["folder"] == "misclassified"][["file", "true", "predicted", "confidence", "analysis"]]

,file,true,predicted,confidence,analysis
24,01_test_aug_0030.jpg,id_2336,id_2970,0.81,Likely cause: cut by the image border. Confide...
25,02_test_aug_0040.jpg,id_2336,id_7007,0.50,"Likely cause: cut by the image border, small f..."
26,03_test_0069.jpg,id_2970,id_7007,0.74,"Clear, unaugmented face. Low confidence (0.74)."
27,04_test_aug_0048.jpg,id_2970,id_4422,0.63,"Likely cause: grey-scale copy, small face (104..."
28,05_test_aug_0013.jpg,id_4422,id_2970,0.62,Likely cause: cut by the image border. id_4422...
29,06_test_aug_0052.jpg,id_4422,id_2970,0.59,"Clear, unaugmented face. id_4422 / id_2970 is ..."
30,07_test_aug_0090.jpg,id_7007,id_4422,0.91,Likely cause: partly covered by a dropout hole...
31,08_test_aug_0082.jpg,id_7007,id_4422,0.84,"Likely cause: cut by the image border, grey-sc..."


---
## Part B: Multi-face images
4 original and 4 augmented images where every face is found and named correctly (most faces first), and the
4 images with the most errors.

In [7]:
# Errors per test image at the selected settings
records = use_identity(predict_split(pipe, de.split_images("test")), pipe.settings["identity"])
rows = []
for r in records:
    a = de.analyze_image(r, pipe.settings["det_conf"])
    errors = sum(o != "TP" for o in a["pred_outcome"]) + sum(o != "found" for o in a["gt_outcome"])
    rows.append(dict(image=r["image"], augmented="_aug_" in r["image"], faces=len(r["gt_classes"]),
                     found=a["gt_outcome"].count("found"), errors=errors))
images = pd.DataFrame(rows)
clean = images[images["errors"] == 0].sort_values(["faces", "image"], ascending=[False, True])
picks = pd.concat([clean[~clean["augmented"]].head(4), clean[clean["augmented"]].head(4),
                   images.sort_values(["errors", "image"], ascending=[False, True]).head(4)])
picks

,image,augmented,faces,found,errors
0,test_0001.jpg,False,4,4,0
9,test_0010.jpg,False,4,4,0
11,test_0012.jpg,False,4,4,0
18,test_0019.jpg,False,4,4,0
109,test_aug_0010.jpg,True,4,4,0
143,test_aug_0044.jpg,True,4,4,0
150,test_aug_0051.jpg,True,4,4,0
167,test_aug_0068.jpg,True,4,4,0
176,test_aug_0077.jpg,True,3,1,4
111,test_aug_0012.jpg,True,4,2,3


In [8]:
# Draw the pipeline's boxes and names on each image
rows, tiles = [], []
for k, r in enumerate(picks.itertuples(), 1):
    path = PROJECT_ROOT / "data" / "detection" / "images" / "test" / r.image
    found = pipe.predict(path)
    image = pipe.annotate(path, found)
    file = f"{k:02d}_{Path(r.image).stem}.jpg"
    image.save(GALLERY_DIR / "multi_face" / file)
    tiles.append(image.resize((320, 320)))
    rows.append(dict(file=file, faces=r.faces, found_correctly=r.found,
                     predicted=", ".join(f["name"] + ("" if f["agree"] else " (?)") for f in found)))
multi = pd.DataFrame(rows)
sheet = Image.new("RGB", (4 * 326, 3 * 326), "white")  # contact sheet, 4 per row
for k, t in enumerate(tiles):
    sheet.paste(t, ((k % 4) * 326, (k // 4) * 326))
sheet.save(GALLERY_DIR / "multi_face.png")
multi

,file,faces,found_correctly,predicted
0,01_test_0001.jpg,4,4,"id_2336, id_7007, id_2970, id_4422"
1,02_test_0010.jpg,4,4,"id_2970, id_2336, id_4422, id_7007"
2,03_test_0012.jpg,4,4,"id_2970 (?), id_2336, id_4422 (?), id_7007 (?)"
3,04_test_0019.jpg,4,4,"id_7007 (?), id_2970, id_4422, id_2336"
4,05_test_aug_0010.jpg,4,4,"id_7007, id_2336, id_4422, id_2970"
5,06_test_aug_0044.jpg,4,4,"id_7007, id_2336, id_4422, id_2970"
6,07_test_aug_0051.jpg,4,4,"id_2970 (?), id_2336, id_4422, id_7007 (?)"
7,08_test_aug_0068.jpg,4,4,"id_2970, id_2336, id_7007, id_4422"
8,09_test_aug_0077.jpg,3,1,"id_2336, id_4422, id_2970 (?)"
9,10_test_aug_0012.jpg,4,2,"id_2970, id_2336, id_2336 (?)"


---
## Part C: Gallery README

In [9]:
# Markdown table with links to the images
def table(df, folder):
    df = df.assign(file=[f"[{f}]({folder}/{f})" for f in df["file"]])
    lines = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


c = single[single["folder"] == "correct"].drop(columns=["folder", "analysis"])
m = single[single["folder"] == "misclassified"].drop(columns="folder")
readme = f"""# Final results gallery (Milestone 4)

Held-out **test** split, generated by [`milestone4/Final_Gallery.ipynb`](../../../milestone4/Final_Gallery.ipynb).
Settings: detector confidence {pipe.settings['det_conf']}, final name from the {pipe.settings['identity']}.
The classifier names {acc:.1%} of the {len(faces)} labelled test faces correctly.

## Correctly classified single faces ({len(c)})

![correct](correct.png)

{table(c, "correct")}

## Misclassified single faces ({len(m)})

![misclassified](misclassified.png)

{table(m, "misclassified")}

## Multi-face images ({len(multi)})

Box colours: id_2336 orange, id_2970 blue, id_4422 green, id_7007 pink. `?` marks a face where the detector and the
classifier give different names.

![multi-face](multi_face.png)

{table(multi, "multi_face")}
"""
(GALLERY_DIR / "README.md").write_text(readme)
print("Saved", GALLERY_DIR / "README.md")

Saved /Users/pmodi/Desktop/P2/Rhea/NU Grad/IE 7615/Project 1/IE7615-Group3-Project1/results/milestone4/gallery/README.md
